In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=10000)

x_train = pad_sequences(x_train, maxlen=200)
x_test = pad_sequences(x_test, maxlen=200)

model = Sequential([
    Embedding(10000, 32),
    SimpleRNN(32),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.fit(x_train, y_train, epochs=3,
          batch_size=64, validation_split=0.2)

loss, accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", accuracy)

reviews = [
    "This product is excellent and very useful",
    "This product is very bad and disappointing"
]

word_index = imdb.get_word_index()

sequences = []
for review in reviews:
    words = review.lower().split()
    sequence = [word_index.get(word, 2) + 3 for word in words]
    sequences.append(sequence)

sequences = pad_sequences(sequences, maxlen=200)

predictions = model.predict(sequences)

for i, prediction in enumerate(predictions):
    sentiment = "Positive" if prediction[0] >= 0.5 else "Negative"
    print(reviews[i], ":", sentiment)

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 32ms/step - accuracy: 0.6274 - loss: 0.6383 - val_accuracy: 0.6566 - val_loss: 0.6643
Epoch 2/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 9s 29ms/step - accuracy: 0.7875 - loss: 0.4603 - val_accuracy: 0.7216 - val_loss: 0.5423
Epoch 3/3
313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 28ms/step - accuracy: 0.8785 - loss: 0.2975 - val_accuracy: 0.7996 - val_loss: 0.4462
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.8091 - loss: 0.4380
Test Accuracy: 0.8091199994087219
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step
This product is excellent and very useful : Positive
This product is very bad and disappointing : Negative


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

text = """deep learning is a part of machine learning.
neural networks are used in artificial intelligence.
recurrent neural networks process sequential data.
deep learning helps computers learn from data."""

chars = sorted(set(text))
char_to_index = {c: i for i, c in enumerate(chars)}
index_to_char = {i: c for i, c in enumerate(chars)}

sequence_length = 20

x = []
y = []

for i in range(len(text) - sequence_length):
    x.append([char_to_index[c] for c in text[i:i+sequence_length]])
    y.append(char_to_index[text[i+sequence_length]])

x = np.array(x)
y = np.array(y)

x = tf.keras.utils.to_categorical(x, num_classes=len(chars))
y = tf.keras.utils.to_categorical(y, num_classes=len(chars))

model = Sequential([
    SimpleRNN(128, input_shape=(sequence_length, len(chars))),
    Dense(len(chars), activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x, y, epochs=100, batch_size=16, verbose=0)

seed = "deep learning is a "
generated = seed

for i in range(100):
    sequence = [char_to_index[c] for c in generated[-sequence_length:]]
    sequence = tf.keras.utils.to_categorical(
        [sequence], num_classes=len(chars)
    )

    prediction = model.predict(sequence, verbose=0)
    next_char = index_to_char[np.argmax(prediction[0])]

    generated += next_char

print("Original Text:")
print(text)

print("\nGenerated Text:")
print(generated)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Original Text:
deep learning is a part of machine learning.
neural networks are used in artificial intelligence.
recurrent neural networks process sequential data.
deep learning helps computers learn from data.

Generated Text:
deep learning is a  art of machine learning.
neural networks are used in artificial intelligence.
recurrent neural netw
